# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object (not as a dict)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available RecordSets and the fields (by @id)
print("Available RecordSets in the dataset:")
for record_set in metadata.record_sets:
    print(f"- RecordSet name: {record_set.name} | @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        columns_str = ', '.join([c.id for c in field.columns]) if hasattr(field, 'columns') else 'N/A'
        print(f"    - Field name: {field.name} | @id: {field.id} | columns @id: {columns_str}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
# Gather the list of record set @id's
record_sets = [rs.id for rs in metadata.record_sets]
print('RecordSet @id list:', record_sets)

dataframes = {}

# For this demonstration, we select the first record set for EDA
selected_record_set_id = record_sets[0]
# Print what we selected
print(f"\nSelected RecordSet @id for analysis: {selected_record_set_id}")

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

if selected_record_set_id in dataframes:
    print(f"Columns in DataFrame for RecordSet {selected_record_set_id}:")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    print(f"Warning: No data for RecordSet {selected_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, automatically select a numeric field (e.g. one containing 'MW' or 'Abundance' or 'Count')
import re

df = dataframes.get(selected_record_set_id, pd.DataFrame())
if not df.empty:
    # Try to guess a numeric field
    numeric_candidate_fields = [col for col in df.columns if re.search(r'(count|abundance|mw|coverage|score|number|percent|intensity|amount|weight|value)', col, re.IGNORECASE)]
    if len(numeric_candidate_fields) == 0:
        # fallback: pick the first numeric-looking column
        for col in df.columns:
            try:
                df[col].astype(float)
                numeric_candidate_fields.append(col)
                break
            except Exception:
                pass
    if numeric_candidate_fields:
        numeric_field = numeric_candidate_fields[0]
        try:
            df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        except Exception:
            pass
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head(8))

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head(8))

        # Try to find a group field among strings
        possible_group_fields = [col for col in df.columns if re.search(r'(group|class|type|sample|condition|accession|id|category|species|protein)', col, re.IGNORECASE)]
        group_field = possible_group_fields[0] if possible_group_fields else None
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head(10))
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data available for the selected RecordSet for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_candidate_fields:
    # Histogram of the numeric field
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Boxplot grouped by group_field if available
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to load, examine, and analyze the 'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells' dataset using the `mlcroissant` library.
- We provided a summary of available record sets and fields (referenced by their Croissant `@id`), and performed basic data exploration including simple EDA and visualization.
- For advanced analysis, users may further explore relationships across multiple record sets or employ statistical/machine learning methods depending on their research question.

**References**:
Dataset Croissant schema: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json